# Notebook 15 — Fannie Mae external dataset construction
Build a **provisional**, unscored loan-level dataset from all eligible actual 2018 Q1 originations in the available acquisition file. The 22 common predictors and 24-predictor FICO sensitivity are frozen proposals, not approval to score. Preserve all earlier notebooks and Freddie artifacts.

## 1. Verify the reviewed contract and source files
Record release evidence if available. An empty release field does not prevent exploratory dataset construction, but blocks scoring.

In [2]:
from pathlib import Path
import ast, hashlib, json, re, zipfile
from collections import Counter
from functools import lru_cache
from datetime import datetime, timezone
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
SOURCE = ROOT/'data/raw/Fannie_Mae/2018Q1/2018Q1.csv'
BASE = ROOT/'data/processed/Freddie_Mac/model_baselines_2015_2017'
FEATURE_SOURCE = ROOT/'src/freddie_features.py'
manifest = json.loads((BASE/'model_manifest.json').read_text(encoding='utf-8'))
contract_file = BASE/'freddie_feature_contract.json'
contract = json.loads(contract_file.read_text(encoding='utf-8'))
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
assert sha(contract_file) == manifest['feature_contract_sha256']
assert manifest['horizon_months'] == 24
constants = {t.id: ast.literal_eval(node.value) for node in ast.parse(FEATURE_SOURCE.read_text(encoding='utf-8')).body if isinstance(node,ast.Assign) for t in node.targets if isinstance(t,ast.Name) and t.id in {'NUMERIC_FEATURES','CATEGORICAL_FEATURES'}}
NUM = constants['NUMERIC_FEATURES']; CAT = constants['CATEGORICAL_FEATURES']
assert NUM+CAT == manifest['predictors'] and len(NUM+CAT)==29
assert set(NUM)==set(contract['numeric_medians']) and set(CAT)==set(contract['category_levels'])
assert contract['training_years']==[2015,2016,2017]
print('Verified frozen contract:', len(NUM), 'numeric and', len(CAT), 'categorical predictors')
import numpy as np
from collections import Counter
from datetime import datetime, timezone
PROCESSED = ROOT/'data/processed/Fannie_Mae'
REVIEW_OVERRIDE = None  # Set to exact 14G ZIP if several different versions exist.
RELEASE_EVIDENCE = ''  # Download date, release identifier, matching glossary (not inferred from file name).
assert SOURCE.stat().st_size == 6928164686, 'Different raw file; review provenance first.'
review_paths = [Path(REVIEW_OVERRIDE)] if REVIEW_OVERRIDE else sorted(PROCESSED.rglob('notebook_14g_final_compatibility_review.zip'))
assert review_paths, 'Locate the reviewed 14G ZIP.'
assert len({sha(p) for p in review_paths}) == 1, 'Set REVIEW_OVERRIDE to the exact reviewed ZIP.'
review_path = review_paths[-1]
with zipfile.ZipFile(review_path) as z:
    proposal = json.loads(z.read('candidate_contract.json'))
assert proposal['frozen_contract_sha256'] == sha(contract_file)
assert proposal['sample_loans'] == 4749 and proposal['sample_rows'] == 245552
assert proposal['training_years'] == [2015,2016,2017]
assert not proposal['models_scored']
common = proposal['original_predictors_retained'] + proposal['additional_candidate']
assert len(common)==22 and len(set(common))==22
assert common[-1]=='PURCHASE_REFINANCE_GROUP'
assert common[:-1] == [c for c in NUM+CAT if c not in proposal['excluded_original']]
print('Reviewed contract:', len(common), 'predictors; FICO sensitivity adds two.')
print('Release evidence supplied:', bool(RELEASE_EVIDENCE))


Verified frozen contract: 15 numeric and 14 categorical predictors
Reviewed contract: 22 predictors; FICO sensitivity adds two.
Release evidence supplied: False


## 2. Define the target and boundary tests
Use reporting month 0–23 relative to first payment. Preserve the Freddie-aligned eligibility rule and the stricter sensitivity. Numeric serious delinquency is a separate outcome; unknown statuses are not recoded as current. Unexpected codes, duplicates, terminal-date conflicts or invalid reporting dates block export.

In [3]:
ADVERSE = {'02','03','09','15'}
KNOWN = ADVERSE | {'01','06','16','96'}
POSITIONS = {'loan_id':2,'report':3,'channel':4,'rate':8,'upb':10,'term':13,'orig':14,'first_payment':15,'ltv':20,'cltv':21,'borrowers':22,'dti':23,'borrower_fico':24,'coborrower_fico':25,'purpose':27,'property_type':28,'units':29,'occupancy':30,'state':31,'mi':34,'amortization':35,'delinq':40,'zero_code':44,'zero_month':45,'mi_type':73}
DATE = re.compile(r'(0[1-9]|1[0-2])(19|20)\d{2}')
def month(v):
    return int(v[2:])*12+int(v[:2])-1 if DATE.fullmatch(v) else None

def target_for(rows):
    orig = {month(r['orig']) for r in rows}
    fp = {month(r['first_payment']) for r in rows}
    result = dict(quarter=None, primary=False, strict=False, serious=False, adverse=False, reason='unresolved_dates', missing_months=None, unknown_active=False, repurchase=False, terminal_timing_disagreements=0)
    reports = [month(r['report']) for r in rows]
    if None in reports or len(set(reports)) != len(reports):
        raise ValueError('Invalid or duplicate reporting month; stop and investigate.')
    if any(r['zero_code'] not in KNOWN|{''} for r in rows):
        raise ValueError('Unexpected terminal code; stop and investigate.')
    if any(not (re.fullmatch(r'\d{1,2}',r['delinq']) or r['delinq'] in {'XX',''}) for r in rows):
        raise ValueError('Unexpected delinquency code; stop and investigate.')
    terminal = [(r,m) for r,m in zip(rows,reports) if r['zero_code']]
    if len({r['zero_code'] for r,m in terminal})>1 or len({r['zero_month'] for r,m in terminal})>1:
        raise ValueError('Conflicting terminal records; review before export.')
    if any(month(r['zero_month']) is None for r,m in terminal):
        raise ValueError('Terminal code without valid effective month.')
    if len(orig)!=1 or len(fp)!=1 or None in orig or None in fp:
        return result
    o=next(iter(orig)); start=next(iter(fp)); end=start+23
    result['quarter']=f'{o//12}Q{(o%12)//3+1}'
    w=[(r,m) for r,m in zip(rows,reports) if start<=m<=end]
    tr=[(r,m) for r,m in w if r['zero_code']]
    left=min(reports)>start
    complete=max(reports)>=end or bool(tr)
    endpoint=min(end,min(m for r,m in tr)) if tr else end
    missing=len(set(range(start,endpoint+1))-{m for r,m in w})
    unknown=any(not r['delinq'].isdigit() and not r['zero_code'] for r,m in w)
    repurchase=any(r['zero_code']=='06' for r,m in tr)
    serious=any(r['delinq'].isdigit() and int(r['delinq'])>=3 for r,m in w)
    adverse=any(r['zero_code'] in ADVERSE for r,m in tr)
    primary=not left and complete
    result.update(primary=primary,strict=primary and missing==0 and not unknown and not repurchase, serious=serious,adverse=adverse,reason='left_truncated' if left else ('incomplete_followup' if not complete else 'eligible'),missing_months=missing,unknown_active=unknown,repurchase=repurchase,terminal_timing_disagreements=sum((start<=m<=end)!=(start<=month(r['zero_month'])<=end) for r,m in terminal))
    return result

def demo(offsets,event=None,terminal=None,unknown=None):
    start=2018*12+2
    def fmt(m): return f'{m%12+1:02d}{m//12:04d}'
    return [dict(orig='012018',first_payment='032018',report=fmt(start+i),delinq='XX' if i==unknown else ('03' if i==event else '00'),zero_code=terminal[1] if terminal and i==terminal[0] else '',zero_month=fmt(start+i) if terminal and i==terminal[0] else '') for i in offsets]
for rows,expected in [(demo(range(24)),(True,True,False)),(demo(range(24),23),(True,True,True)),(demo(range(25),24),(True,True,False)),(demo(range(7),terminal=(6,'09'),unknown=6),(True,True,True)),(demo(range(7),terminal=(6,'06'),unknown=6),(True,False,False)),(demo(range(1,24)),(False,False,False)),(demo(range(12)),(False,False,False)),(demo([i for i in range(24) if i!=5]),(True,False,False)),(demo(range(24),unknown=5),(True,False,False))]:
    r=target_for(rows)
    assert (r['primary'],r['strict'],r['serious'] or r['adverse'])==expected
print('Nine synthetic target checks passed.')

Nine synthetic target checks passed.


## 3. Stream complete loan histories
Read each physical record once. This implementation requires contiguous loan IDs; it checks that assumption for the whole file and stops if a loan reappears. Only earliest reporting snapshots and loan-level audit results are retained for actual 2018 Q1 originations. A hash subsample reconciles the earlier diagnostic run. No full monthly dataset is held in memory.

In [4]:
signature=(SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)
source_hash=hashlib.sha256()
seen=set(); sample_counts=Counter(); scan=Counter(); cohort_counts=Counter(); reasons=Counter(); records=[]
current=None; history=[]
indexes=[i-1 for i in POSITIONS.values()]

def finish(loan_id, rows):
    if loan_id is None: return
    if loan_id in seen: raise ValueError('Noncontiguous loan histories. A partitioned scan is required; no dataset exported.')
    seen.add(loan_id)
    a=target_for(rows)
    quarter=a['quarter'] or 'unresolved'
    cohort_counts[quarter]+=1
    reasons[(quarter,a['reason'])]+=1
    if int.from_bytes(hashlib.blake2b(loan_id,digest_size=8,person=b'Fannie14v1').digest(),'big')%100==0:
        sample_counts['loans']+=1; sample_counts['rows']+=len(rows)
        for policy in ['primary','strict']:
            if a[policy]:
                sample_counts[policy]+=1
                sample_counts[policy+'_events']+=int(a['serious'] or a['adverse'])
    if quarter=='2018Q1':
        first=min(rows,key=lambda r:month(r['report']))
        records.append({**first,**a,'history_rows':len(rows), 'origination_field_conflict_count':sum(len({r[k] for r in rows if r[k]})>1 for k in ['rate','upb','term','ltv','cltv','dti','borrower_fico','coborrower_fico','mi','mi_type','purpose','channel','units','occupancy','state','property_type','amortization','borrowers'])})

with SOURCE.open('rb') as stream:
    for i,raw in enumerate(stream,1):
        source_hash.update(raw)
        parts=raw.rstrip(b'\r\n').split(b'|')
        if len(parts)!=113: raise ValueError(f'Unexpected physical field count at row {i}')
        key=parts[1].strip()
        if not re.fullmatch(rb'[A-Za-z0-9]{1,12}',key): raise ValueError(f'Invalid loan ID at row {i}')
        if key!=current:
            finish(current,history)
            current=key; history=[]
        history.append(dict(zip(POSITIONS,(parts[j].strip().decode('ascii') for j in indexes))))
        if i%2_000_000==0: print(f'{i:,} rows scanned; {len(seen):,} completed loans')
    finish(current,history)
scan['physical_rows']=i; scan['unique_loans']=len(seen)
assert signature==(SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)
assert i==23563399
assert dict(sample_counts)=={'loans':4749,'rows':245552,'primary':4491,'strict':4488,'primary_events':33,'strict_events':33}, sample_counts
assert sum(cohort_counts.values())==len(seen)
first=pd.DataFrame(records)
assert not first.empty and first.loan_id.is_unique
assert len(first)==cohort_counts['2018Q1']
print(dict(scan)); print('Diagnostic sample reconciled:',dict(sample_counts)); print('Actual 2018Q1 originations:',len(first))

2,000,000 rows scanned; 38,849 completed loans
4,000,000 rows scanned; 77,843 completed loans
6,000,000 rows scanned; 116,863 completed loans
8,000,000 rows scanned; 155,865 completed loans
10,000,000 rows scanned; 194,667 completed loans
12,000,000 rows scanned; 233,636 completed loans
14,000,000 rows scanned; 272,604 completed loans
16,000,000 rows scanned; 311,729 completed loans
18,000,000 rows scanned; 350,843 completed loans
20,000,000 rows scanned; 389,651 completed loans
22,000,000 rows scanned; 428,514 completed loans
{'physical_rows': 23563399, 'unique_loans': 458830}
Diagnostic sample reconciled: {'loans': 4749, 'rows': 245552, 'primary': 4491, 'primary_events': 33, 'strict': 4488, 'strict_events': 33}
Actual 2018Q1 originations: 260395


## 4. Construct candidate features
Use exactly the reviewed 14G transformations. Do not use future reporting records to fill the earliest snapshot. Report inconsistent MI and feature conflicts; they are not silently approved.

In [5]:
import numpy as np

def valid_number(s, low, high):
    v = pd.to_numeric(s, errors='coerce')
    return v.where(v.between(low, high))

def harmonize_mi(percent, kind):
    v = valid_number(percent, 1, 55)
    no_mi = percent.eq('') & kind.eq('')
    insured = kind.isin(['1','2','3'])
    inconsistent = ~(no_mi | (insured & (percent.eq('') | v.notna())))
    mapped = v.where(insured).mask(no_mi, 0.0)
    return mapped, inconsistent

mi, mi_bad = harmonize_mi(first.mi, first.mi_type)
primary_score = valid_number(first.borrower_fico,300,850)
co_score = valid_number(first.coborrower_fico,300,850)
min_score = pd.concat([primary_score,co_score],axis=1).min(axis=1)
x = pd.DataFrame(index=first.index)
dti = valid_number(first.dti,1,998)
ltv = valid_number(first.ltv,1,998)
cltv = valid_number(first.cltv,1,998)
upb = pd.to_numeric(first.upb,errors='coerce')
upb = upb.where(upb.gt(0))
x['ORIGINAL_DTI'] = dti
x['ORIGINAL_LTV'] = ltv
x['ORIGINAL_INTEREST_RATE'] = valid_number(first.rate,0.01,100)
x['LOG_ORIGINAL_UPB'] = np.log1p(upb)
x['CLTV_MINUS_LTV'] = (cltv-ltv).clip(lower=0)
x['HAS_SUBORDINATE_FINANCING'] = cltv.gt(ltv).astype('int8')
x['HIGH_LTV_GT_80'] = ltv.gt(80).astype('int8')
for name,s in [('ORIGINAL_DTI',dti),('ORIGINAL_LTV',ltv),('CLTV',cltv)]:
    x[name+'_MISSING'] = s.isna().astype('int8')
x['MI_PERCENT'] = mi
x['HAS_MORTGAGE_INSURANCE'] = mi.gt(0).astype('int8')
x['MI_PERCENT_MISSING'] = mi.isna().astype('int8')
for name,field,allowed in [('NUMBER_OF_UNITS','units',{'1','2','3','4'}),('OCCUPANCY_STATUS','occupancy',{'P','S','I'}),('CHANNEL','channel',{'B','C','R'}),('AMORTIZATION_TYPE','amortization',{'FRM','ARM'}),('PROPERTY_TYPE','property_type',{'SF','PU','CO','CP','MH'})]:
    x[name] = first[field].where(first[field].isin(allowed))
x['PROPERTY_STATE'] = first.state.where(first.state.str.fullmatch('[A-Z]{2}'))
x['ORIGINAL_LOAN_TERM'] = first.term.where(first.term.str.fullmatch('[0-9]+'))
b = pd.to_numeric(first.borrowers,errors='coerce')
x['NUMBER_OF_BORROWERS_GROUP'] = pd.Series(pd.NA,index=first.index,dtype='string').mask(b.eq(1),'1').mask(b.between(2,10),'2+')
x['PURCHASE_REFINANCE_GROUP'] = first.purpose.map({'P':'PURCHASE','C':'REFINANCE','R':'REFINANCE'})
# Apply equivalent P -> PURCHASE, C/N -> REFINANCE on Freddie in the later dataset notebook, only after verifying Freddie purpose codes.
assert set(x)==set(common)
x=x[common].copy()
x['FICO_SCORE']=min_score
x['FICO_SCORE_MISSING']=min_score.isna().astype('int8')
assert len(x.columns)==24
print('Candidate predictors constructed:',len(x.columns))


Candidate predictors constructed: 24


## 5. Validate loan-level labels and export locally
Keep all actual 2018 Q1 originations for audit, with nullable labels for ineligible loans. The primary and strict sensitivity labels are separate. The 24-predictor proxy is not a verified equivalent to Freddie FICO. The ZIP contains aggregate evidence only; the local Parquet is not for GitHub.

In [6]:
out_data=pd.concat([first[['loan_id','orig','first_payment','report','primary','strict','reason','missing_months','unknown_active','repurchase','terminal_timing_disagreements','origination_field_conflict_count']].copy(),x],axis=1)
out_data['FICO_PRIMARY_BORROWER_PROXY']=primary_score
out_data['MI_MAPPING_INCONSISTENT']=mi_bad
out_data['ORIGINATION_COHORT']='2018Q1'
for policy in ['primary','strict']:
    eligible=first[policy]
    for name,flag in [('COMPOSITE',first.serious|first.adverse),('NUMERIC_90_PLUS',first.serious)]:
        out_data[f'TARGET_{policy.upper()}_{name}']=flag.astype('Int8').where(eligible,pd.NA)
assert out_data.loan_id.is_unique and (first.strict<=first.primary).all()
assert not out_data.loc[~first.primary,'TARGET_PRIMARY_COMPOSITE'].notna().any()
assert out_data.loc[first.strict,'TARGET_PRIMARY_COMPOSITE'].equals(out_data.loc[first.strict,'TARGET_STRICT_COMPOSITE'])
summary=[]
for policy in ['primary','strict']:
    e=first[policy]; total=int(e.sum()); events=int((e&(first.serious|first.adverse)).sum())
    summary.append(dict(policy=policy,total=len(first),eligible=total,ineligible=len(first)-total,composite=events,numeric_serious=int((e&first.serious).sum()),adverse=int((e&first.adverse).sum()),event_rate=events/total if total else None))
summary=pd.DataFrame(summary)
assert (summary.eligible>0).all()
profiles=pd.DataFrame([dict(predictor=c,missing=int(x.loc[first.primary,c].isna().sum()),distinct=int(x.loc[first.primary,c].nunique())) for c in x])
quality=pd.DataFrame([dict(check=k,affected_loans=int(v.sum())) for k,v in {'mi_mapping_inconsistent':mi_bad,'origination_field_conflicts':first.origination_field_conflict_count.gt(0),'terminal_window_disagreement':first.terminal_timing_disagreements.gt(0),'unknown_active':first.unknown_active,'repurchase':first.repurchase}.items()])
gates=pd.DataFrame([('release_provenance','pending_review'),('target_semantic_equivalence','pending_review'),('population_coverage','limited_single_acquisition_file'),('feature_contract_approval','pending_review'),('model_scoring','blocked')],columns=['gate','status'])
out=PROCESSED/'notebook_15_external_dataset_construction'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out.mkdir(parents=True,exist_ok=False)
local=out/'fannie_2018q1_candidate_dataset.parquet'
out_data.to_parquet(local,index=False)
readback=pd.read_parquet(local)
pd.testing.assert_frame_equal(out_data,readback,check_dtype=False)
metadata=dict(source=str(SOURCE),source_sha256=source_hash.hexdigest(),source_bytes=signature[0],physical_rows=i,unique_loans=len(seen),review_14g_sha256=sha(review_path),frozen_contract_sha256=sha(contract_file),sample_reconciliation=dict(sample_counts),release_evidence=RELEASE_EVIDENCE,release_verified=False,final_approved=False,models_scored=False,candidate_dataset_sha256=sha(local),primary_features=common,fico_sensitivity_features=common+['FICO_SCORE','FICO_SCORE_MISSING'],scope='Actual 2018Q1 originations in 2018Q1 acquisition file only; not complete origination cohort.')
for name,table in {'target_summary.csv':summary,'feature_profile.csv':profiles,'quality_checks.csv':quality,'review_gates.csv':gates,'origination_counts.csv':pd.DataFrame(cohort_counts.items(),columns=['cohort','loans']),'eligibility_reasons.csv':pd.DataFrame([(q,r,v) for (q,r),v in reasons.items()],columns=['cohort','reason','loans'])}.items(): table.to_csv(out/name,index=False)
(out/'metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
review=out/'notebook_15_external_dataset_review.zip'
with zipfile.ZipFile(review,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()):
        if f.suffix in {'.csv','.json'}: z.write(f,f.name)
display(summary); display(quality); display(gates)
print('Local candidate dataset:',local); print('Upload only:',review)

,policy,total,eligible,ineligible,composite,numeric_serious,adverse,event_rate
0,primary,260395,260244,151,1960,1958,95,0.007531
1,strict,260395,260043,352,1947,1945,95,0.007487


,check,affected_loans
0,mi_mapping_inconsistent,0
1,origination_field_conflicts,56
2,terminal_window_disagreement,0
3,unknown_active,0
4,repurchase,200


,gate,status
0,release_provenance,pending_review
1,target_semantic_equivalence,pending_review
2,population_coverage,limited_single_acquisition_file
3,feature_contract_approval,pending_review
4,model_scoring,blocked


Local candidate dataset: e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_15_external_dataset_construction\20261124T164838809452Z\fannie_2018q1_candidate_dataset.parquet
Upload only: e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_15_external_dataset_construction\20261124T164838809452Z\notebook_15_external_dataset_review.zip


## Review gate
Upload only the aggregate ZIP. Do not commit the Parquet or score any model. The dataset is provisional until release documentation, target semantics, feature mapping and comparison population are approved.